In [48]:
import sys

sys.path.append("../src/")
sys.path.append("../src/AtomicTrit")

from AtomicTrit import elastic
from AtomicTrit import constants
from AtomicTrit import potentials
import numpy as np
import pylab as plt
import pandas as pd
import json
from scipy.interpolate import interp1d
#====================
# HeH potentials
#====================
path="../src/AtomicTrit/"
datLong=pd.read_csv(path+"InputData/HeH_MeyerFromhold.csv")
datShort=pd.read_csv(path+"InputData/HeH_ShortCorrection.csv")


In [49]:
#Here are the masses and atom names we will need
muH=constants.HydrogenConstants.mu
m4He_ov_mH=4.0026/1.008
m3He_ov_mH=3.0160/1.008
mT_ov_mH=constants.TritiumConstants.mu/constants.HydrogenConstants.mu
mD_ov_mH=2.0141/1.008
kB=8.617e-5


HAtoms=["H","D","T"]
HeAtoms=["3He","4He"]

HMasses=[1,2.0141/1.008,constants.TritiumConstants.mu/constants.HydrogenConstants.mu]
HeMasses=[3.0160/1.008,4.0026/1.008]

MuBySpecies={}
for Hi in range(0,len(HMasses)):
    for Hei in range(0,len(HeMasses)):
        muC=2.*HMasses[Hi]*HeMasses[Hei]/(HMasses[Hi]+HeMasses[Hei])
        name= HAtoms[Hi]+"-"+HeAtoms[Hei]
        MuBySpecies[name]=muC



In [50]:
#Here we make the potential functions

bohr = constants.BohrInAng/constants.hcInEVAngstrom

interpMF= interp1d(datLong.R*bohr,datLong.Total*constants.HartreeInEV*1e-6, kind='cubic', bounds_error=False,fill_value='extrapolate')

interpShort = interp1d(datShort.R*bohr,datShort.Modified*constants.HartreeInEV*1e-6, kind='cubic', bounds_error=False,fill_value='extrapolate')

def ExtendedMF(R):
    return potentials.VanDerWaalsExtension(R,interpMF,15*bohr)

def ModifiedMF(R):
    return potentials.CompositePotential(R, [interpShort, ExtendedMF], [0, 6.0 * bohr, 1e6])


In [ ]:
# Calc on log energy scale

TotalXBySpecies={}
MuBySpecies={}
for Hi in range(0,len(HMasses)):
    for Hei in range(0,len(HeMasses)):
        muC=2.*HMasses[Hi]*HeMasses[Hei]/(HMasses[Hi]+HeMasses[Hei])
        name= HAtoms[Hi]+"-"+HeAtoms[Hei]

        k_eV  = np.logspace(0, np.log10(4*constants.hcInEVAngstrom), 100)
        k_A = k_eV / constants.hcInEVAngstrom


        r0       = 1e-9
        intlimit = 100 * constants.BohrInAng/constants.hcInEVAngstrom
        rhos = np.linspace(r0, intlimit, 100)



        ls = np.arange(0, 10, 1)

        sigma_T_by_l = {}
        sigma_T_total = np.zeros_like(k_A)   # activate if you want Σ_l

        for l in ls:
            sigma_T_partial = np.array([
                elastic.GetCrossSection(rhos, k, l+1e-4, muC*muH,
                                        ModifiedMF, 'Radau')/2
                for k in k_eV
            ])
            sigma_T_by_l[l] = sigma_T_partial
            sigma_T_total   += sigma_T_by_l[l]
        TotalXBySpecies[name]=sigma_T_total
        MuBySpecies[name]=muC

In [6]:
#Calculate it for the other potential

TotalXBySpeciesMF={}
MuBySpecies={}
for Hi in range(0,len(HMasses)):
    for Hei in range(0,len(HeMasses)):
        muC=2.*HMasses[Hi]*HeMasses[Hei]/(HMasses[Hi]+HeMasses[Hei])
        name= HAtoms[Hi]+"-"+HeAtoms[Hei]

        k_eV  = np.logspace(0, np.log10(4*constants.hcInEVAngstrom), 100)
        k_A = k_eV / constants.hcInEVAngstrom

        r0       = 1e-9
        intlimit = 100 * constants.BohrInAng/constants.hcInEVAngstrom
        rhos = np.linspace(r0, intlimit, 100)



        ls = np.arange(0, 10, 1)

        sigma_T_by_l = {}
        sigma_T_total = np.zeros_like(k_A)   # activate if you want Σ_l

        for l in ls:
            sigma_T_partial = np.array([
                elastic.GetCrossSection(rhos, k, l+1e-4, muC*muH,
                                        ExtendedMF, 'Radau')/2
                for k in k_eV
            ])
            sigma_T_by_l[l] = sigma_T_partial
            sigma_T_total   += sigma_T_by_l[l]
        TotalXBySpeciesMF[name]=sigma_T_total
        MuBySpecies[name]=muC

In [7]:
E=k_eV**2/(2*muC*muH)

AllEs=[]
HIso=[]
HeIso=[]
SigmaMF=[]
SigmaModMF=[]

for spec in TotalXBySpecies.keys():
    for i in range(0,len(TotalXBySpecies[spec])):
        HIso.append(spec.split("-")[0])
        HeIso.append(spec.split("-")[1])
        AllEs.append(E[i]/kB)
        SigmaModMF.append(TotalXBySpecies[spec][i])
        SigmaMF.append(TotalXBySpeciesMF[spec][i])

In [8]:

df = pd.DataFrame({
                    'E':AllEs,
                    'HIso':HIso,
                    'HeIso':HeIso,
                    'sigmaMF':SigmaModMF,
                    'sigmaModMF':SigmaMF
                   })
df.to_csv('Tables/HeH_CrossSections.csv')